# 📊 Engineering Placement — Exploratory Data Analysis
**Project:** AI-Powered Engineering Placement Prediction System  
**Developed by:** Ananaya Arora  
**Dataset:** placement.csv (merged engineering features + placement targets)  
**Objective:** Understand distributions, correlations, and key patterns before model training.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14

DATA_PATH = os.path.join('..', 'data', 'placement.csv')
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Column Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Stats ===')
df.describe()

## 2. Target Variable — Placement Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['placement_status'].value_counts()
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#6366f1', '#f97316'], startangle=90)
axes[0].set_title('Placement Distribution')

sns.countplot(data=df, x='placement_status', palette=['#6366f1', '#f97316'], ax=axes[1])
axes[1].set_title('Placement Count')
axes[1].set_xlabel('Status')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

placement_rate = (df['placement_status'] == 'Placed').mean() * 100
print(f'Placement Rate: {placement_rate:.1f}%')

## 3. Placement Rate by Branch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(data=df, x='branch', hue='placement_status',
              palette={'Placed': '#6366f1', 'Not Placed': '#f97316'}, ax=axes[0])
axes[0].set_title('Branch vs Placement')
axes[0].legend(title='Status')

rate = df.groupby('branch')['placement_status'].apply(lambda x: (x == 'Placed').mean() * 100)
rate.sort_values(ascending=False).plot(kind='bar', color='#6366f1', ax=axes[1], rot=0)
axes[1].set_title('Placement Rate by Branch (%)')
axes[1].set_ylabel('Placement Rate (%)')
axes[1].set_ylim(0, 100)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width() / 2, p.get_height() + 1),
                     ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 4. Gender Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='gender', hue='placement_status',
              palette={'Placed': '#6366f1', 'Not Placed': '#f97316'}, ax=axes[0])
axes[0].set_title('Gender vs Placement')
axes[0].legend(title='Status')

rate_g = df.groupby('gender')['placement_status'].apply(lambda x: (x == 'Placed').mean() * 100)
rate_g.plot(kind='bar', color=['#6366f1', '#f97316'], ax=axes[1], rot=0)
axes[1].set_title('Placement Rate by Gender (%)')
axes[1].set_ylabel('Placement Rate (%)')
axes[1].set_ylim(0, 100)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width() / 2, p.get_height() + 1),
                     ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 5. Academic Score Distributions

In [ ]:
num_cols = ['cgpa', 'tenth_percentage', 'twelfth_percentage', 'attendance_percentage']
labels   = ['CGPA', '10th %', '12th %', 'Attendance %']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(num_cols, labels)):
    for status, color in [('Placed', '#6366f1'), ('Not Placed', '#f97316')]:
        subset = df[df['placement_status'] == status][col]
        axes[i].hist(subset, bins=20, alpha=0.6, color=color, label=status)
    axes[i].set_title(f'{label} Distribution')
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.suptitle('Academic Score Distributions by Placement Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Skill Ratings vs Placement

In [ ]:
skill_cols = ['coding_skill_rating', 'communication_skill_rating', 'aptitude_skill_rating']
skill_labels = ['Coding Skill', 'Communication Skill', 'Aptitude Skill']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, label in zip(axes, skill_cols, skill_labels):
    rate = df.groupby(col)['placement_status'].apply(lambda x: (x == 'Placed').mean() * 100)
    rate.plot(kind='bar', color='#6366f1', ax=ax, rot=0)
    ax.set_title(f'{label} vs Placement Rate')
    ax.set_ylabel('Placement Rate (%)')
    ax.set_xlabel(f'{label} (1-5)')
    ax.set_ylim(0, 100)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.0f}%',
                    (p.get_x() + p.get_width() / 2, p.get_height() + 1),
                    ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Internship & Project Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

intern_rate = df.groupby('internships_completed')['placement_status'].apply(lambda x: (x == 'Placed').mean() * 100)
intern_rate.plot(kind='bar', color='#6366f1', ax=axes[0], rot=0)
axes[0].set_title('Internships Completed vs Placement Rate')
axes[0].set_ylabel('Placement Rate (%)')
axes[0].set_xlabel('Internships Completed')

proj_rate = df.groupby('projects_completed')['placement_status'].apply(lambda x: (x == 'Placed').mean() * 100)
proj_rate.plot(kind='line', marker='o', color='#f97316', ax=axes[1])
axes[1].set_title('Projects Completed vs Placement Rate')
axes[1].set_ylabel('Placement Rate (%)')
axes[1].set_xlabel('Projects Completed')

plt.tight_layout()
plt.show()

## 8. Backlogs Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='backlogs', hue='placement_status',
              palette={'Placed': '#6366f1', 'Not Placed': '#f97316'}, ax=axes[0])
axes[0].set_title('Backlogs vs Placement')

backlog_rate = df.groupby('backlogs')['placement_status'].apply(lambda x: (x == 'Placed').mean() * 100)
backlog_rate.plot(kind='bar', color='#ef4444', ax=axes[1], rot=0)
axes[1].set_title('Placement Rate by Backlog Count (%)')
axes[1].set_ylabel('Placement Rate (%)')

plt.tight_layout()
plt.show()

## 9. Salary Analysis

In [ ]:
placed_df = df[df['placement_status'] == 'Placed'].copy()
placed_df = placed_df[placed_df['salary_lpa'] > 0]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(placed_df['salary_lpa'], bins=25, color='#6366f1', alpha=0.8, edgecolor='white')
axes[0].set_title('Salary Distribution (LPA)')
axes[0].set_xlabel('Salary (LPA)')
axes[0].set_ylabel('Frequency')

axes[1].scatter(placed_df['cgpa'], placed_df['salary_lpa'], alpha=0.3, color='#6366f1', s=10)
axes[1].set_title('CGPA vs Salary')
axes[1].set_xlabel('CGPA')
axes[1].set_ylabel('Salary (LPA)')

branch_salary = placed_df.groupby('branch')['salary_lpa'].mean().sort_values(ascending=False)
branch_salary.plot(kind='bar', color='#f97316', ax=axes[2], rot=0)
axes[2].set_title('Average Salary by Branch')
axes[2].set_ylabel('Avg Salary (LPA)')

plt.tight_layout()
plt.show()

print(f'Mean Salary  : {placed_df["salary_lpa"].mean():.2f} LPA')
print(f'Median Salary: {placed_df["salary_lpa"].median():.2f} LPA')
print(f'Max Salary   : {placed_df["salary_lpa"].max():.2f} LPA')
print(f'Min Salary   : {placed_df["salary_lpa"].min():.2f} LPA')

## 10. Correlation Heatmap

In [ ]:
num_df = df[['cgpa', 'tenth_percentage', 'twelfth_percentage', 'backlogs',
             'attendance_percentage', 'projects_completed', 'internships_completed',
             'coding_skill_rating', 'communication_skill_rating', 'aptitude_skill_rating']].copy()
num_df['placed'] = (df['placement_status'] == 'Placed').astype(int)

corr = num_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu',
            center=0, mask=mask, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 11. Box Plots — Score by Placement Status

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
cols = ['cgpa', 'tenth_percentage', 'twelfth_percentage', 'attendance_percentage']
labels = ['CGPA', '10th %', '12th %', 'Attendance %']

for ax, col, label in zip(axes, cols, labels):
    groups = [df[df['placement_status'] == s][col].values for s in ['Placed', 'Not Placed']]
    bp = ax.boxplot(groups, labels=['Placed', 'Not Placed'], patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    bp['boxes'][0].set_facecolor('#6366f1')
    bp['boxes'][1].set_facecolor('#f97316')
    ax.set_title(label)
    ax.set_ylabel(label)

plt.suptitle('Score Distribution by Placement Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Key Insights Summary

In [ ]:
print('=' * 55)
print('  KEY EDA INSIGHTS')
print('=' * 55)

placed   = df[df['placement_status'] == 'Placed']
not_placed = df[df['placement_status'] == 'Not Placed']

print(f'\n  Placement Rate        : {len(placed)/len(df)*100:.1f}%')
print(f'  Placed count          : {len(placed)}')
print(f'  Not Placed count      : {len(not_placed)}')

for col, label in zip(['cgpa','tenth_percentage','twelfth_percentage','attendance_percentage'],
                      ['CGPA','10th%','12th%','Attendance%']):
    p_mean = placed[col].mean()
    np_mean = not_placed[col].mean()
    print(f'  {label:<14} Placed: {p_mean:.2f}  Not Placed: {np_mean:.2f}  Diff: +{p_mean-np_mean:.2f}')

print(f'\n  Avg Backlogs (Placed)    : {placed["backlogs"].mean():.2f}')
print(f'  Avg Backlogs (Not Placed): {not_placed["backlogs"].mean():.2f}')
print(f'  Avg Internships (Placed) : {placed["internships_completed"].mean():.2f}')

print('\n  Top Branch by Placement Rate:', df.groupby('branch')['placement_status'].apply(lambda x: (x=='Placed').mean()).idxmax())
print(f'  Avg Placed Salary           : {placed[placed["salary_lpa"]>0]["salary_lpa"].mean():.2f} LPA')
print('\n' + '=' * 55)